# Lab 5B: CloudWatch Logs Monitoring

In this lab, you'll **debug and troubleshoot** your Amazon SageMaker workloads using Amazon CloudWatch Logs. You'll access training job logs, analyze inference endpoint logs in near real-time, search for error patterns programmatically, trigger and diagnose a real inference error, and learn structured-logging and retention best practices.

## Prerequisites

- Completed **Lab 5A** (CloudWatch Operational Monitoring)
- An active `bank-marketing-*` endpoint from Lab 3A (or historical training-job logs)

## What you'll do

1. Discover SageMaker log groups and streams
2. Read and search **training job logs**
3. Watch **endpoint logs** while generating traffic
4. **Trigger a real inference error** and find it in the logs
5. Analyze log patterns programmatically (error counts, timelines)
6. Set up **Logs Insights queries, metric filters, and log-based alarms** (console)
7. Apply **structured logging** best practices
8. Configure **log retention** for cost and compliance

> ℹ️ **Permissions note**: your notebook execution role has **read-only** CloudWatch Logs access scoped to `/aws/sagemaker/*` (`DescribeLogGroups`, `DescribeLogStreams`, `GetLogEvents`, `FilterLogEvents`). Write operations — Logs Insights queries, metric filters, retention settings — are done in the **console** (where you have full access), and this notebook gives you exact click-paths for those parts.


## Section 1: Setup and Log Group Discovery

SageMaker automatically sends logs to CloudWatch Logs:

| Workload | Log group | Log streams |
|---|---|---|
| Training jobs | `/aws/sagemaker/TrainingJobs` | one per instance, e.g. `<job-name>/algo-1-<epoch>` |
| Endpoints | `/aws/sagemaker/Endpoints/<EndpointName>` | one per instance, e.g. `AllTraffic/i-0abc...` |

Log content includes your script's stdout/stderr, framework logs (XGBoost/PyTorch), container startup/shutdown messages, and stack traces.


In [ ]:
import boto3
import time
import json
import pandas as pd
from collections import Counter
from datetime import datetime, timedelta, timezone

session = boto3.session.Session()
region = session.region_name
logs = session.client('logs')
sm = session.client('sagemaker')
smr = session.client('sagemaker-runtime')

print(f'Region: {region}')
print('Clients ready: logs, sagemaker, sagemaker-runtime')

In [ ]:
# 🔗 Console deep-link helpers — build clickable AWS console URLs from this notebook
import urllib.parse
from IPython.display import Markdown, display

_CW = f'https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}'
_SM = f'https://{region}.console.aws.amazon.com/sagemaker/home?region={region}'

def _star_escape(s):
    """CloudWatch graph/Insights fragment encoding: percent-encode everything, then % → * (lowercase hex)."""
    q = urllib.parse.quote(str(s), safe='')
    out, i = '', 0
    while i < len(q):
        if q[i] == '%':
            out += '*' + q[i+1:i+3].lower(); i += 3
        else:
            out += q[i]; i += 1
    return out

def _dollar_escape(s):
    """CloudWatch Logs fragment encoding: double percent-encode, then % → $."""
    return urllib.parse.quote(urllib.parse.quote(str(s), safe='')).replace('%', '$')

def url_metrics_graph(title, *metrics, stat='Average', period=60):
    """Pre-built CloudWatch metrics graph. metrics: tuples of (namespace, metric_name, dims_dict)."""
    m_parts = []
    for ns, mn, dims in metrics:
        p = [f"~'{_star_escape(ns)}", f"~'{_star_escape(mn)}"]
        for k, v in dims.items():
            p += [f"~'{_star_escape(k)}", f"~'{_star_escape(v)}"]
        m_parts.append('~(' + ''.join(p) + ')')
    graph = (f"~(metrics~({''.join(m_parts)})~view~'timeSeries~stacked~false"
             f"~region~'{region}~stat~'{_star_escape(stat)}~period~{period}~title~'{_star_escape(title)})")
    return f'{_CW}#metricsV2:graph={graph}'

def url_log_group(group, stream=None, filter_pattern=None):
    """CloudWatch Logs group (or stream) page; optional pre-set search filter."""
    u = f'{_CW}#logsV2:log-groups/log-group/{_dollar_escape(group)}'
    if stream:
        u += f'/log-events/{_dollar_escape(stream)}'
    if filter_pattern:
        u += f'$3FfilterPattern$3D{_dollar_escape(filter_pattern)}'
    return u

def url_logs_insights(query, group, hours=1):
    """Logs Insights with the query editor and log group pre-filled."""
    qd = (f"~(end~0~start~-{hours*3600}~timeType~'RELATIVE~unit~'seconds"
          f"~editorString~'{_star_escape(query)}~source~(~'{_star_escape(group)}))")
    return f'{_CW}#logsV2:logs-insights$3FqueryDetail$3D{qd}'

def url_dashboard(name):
    return f'{_CW}#dashboards/dashboard/{urllib.parse.quote(name, safe="")}'

def url_alarm(name):
    return f'{_CW}#alarmsV2:alarm/{urllib.parse.quote(name, safe="")}'

def url_sagemaker(kind, name):
    """kind: 'jobs' (training jobs) or 'endpoints'."""
    return f'{_SM}#/{kind}/{urllib.parse.quote(name, safe="")}'

def url_cloudtrail(**filters):
    """CloudTrail Event history, optionally pre-filtered (EventSource=..., EventName=...)."""
    q = urllib.parse.urlencode(filters)
    return (f'https://{region}.console.aws.amazon.com/cloudtrailv2/home?region={region}#/events'
            + (f'?{q}' if q else ''))

def url_eventbridge_rule(name):
    return (f'https://{region}.console.aws.amazon.com/events/home?region={region}'
            f'#/eventbus/default/rules/{urllib.parse.quote(name, safe="")}')

def url_sns_topic(arn):
    return f'https://{region}.console.aws.amazon.com/sns/v3/home?region={region}#/topic/{arn}'

def show_links(items):
    """Render a clickable link list. items: list of (label, url)."""
    display(Markdown('**🔗 Open in AWS Console:**\n' + '\n'.join(f'- [{l}]({u})' for l, u in items)))

print('Console deep-link helpers loaded ✅')

In [ ]:
# Discover SageMaker log groups
resp = logs.describe_log_groups(logGroupNamePrefix='/aws/sagemaker/')
log_groups = resp['logGroups']

print('SageMaker log groups:')
for lg in log_groups:
    size_mb = lg.get('storedBytes', 0) / 1e6
    retention = lg.get('retentionInDays', 'Never expire')
    print(f"  {lg['logGroupName']:70s} {size_mb:8.2f} MB  retention: {retention}")

In [ ]:
# Pick the resources we'll inspect: latest bank-marketing training job + InService endpoint
tjs = sm.list_training_jobs(SortBy='CreationTime', SortOrder='Descending', MaxResults=10)['TrainingJobSummaries']
training_job_name = next(
    (t['TrainingJobName'] for t in tjs if t['TrainingJobName'].startswith('bank-marketing')),
    tjs[0]['TrainingJobName'] if tjs else None,
)

eps = sm.list_endpoints(SortBy='CreationTime', SortOrder='Descending')['Endpoints']
endpoint_name = next(
    (e['EndpointName'] for e in eps
     if e['EndpointName'].startswith('bank-marketing') and e['EndpointStatus'] == 'InService'),
    next((e['EndpointName'] for e in eps if e['EndpointStatus'] == 'InService'), None),
)

print(f'Training job : {training_job_name}')
print(f'Endpoint     : {endpoint_name or "⚠️ none InService — endpoint sections will be skipped"}')

TRAINING_LOG_GROUP = '/aws/sagemaker/TrainingJobs'
ENDPOINT_LOG_GROUP = f'/aws/sagemaker/Endpoints/{endpoint_name}' if endpoint_name else None

## Section 2: Access and Analyze Training Job Logs

Every training job writes one log stream per instance. The typical structure of a successful run:

```
[ts] Starting training job              ← container startup
[ts] Downloading training data ...      ← data channel setup
[ts] Invoking user training script      ← your script begins
[ts] [INFO] metrics, epochs, ...        ← your print/logging output
[ts] Reporting training SUCCESS         ← completion
```

Failure signatures to know:
- `MemoryError: Unable to allocate ...` → instance too small
- `NoSuchKey` / `Access Denied` on S3 → wrong data path or IAM role permissions
- `ModuleNotFoundError` → missing dependency in `requirements.txt`


In [ ]:
# Find the log streams for our training job
streams = logs.describe_log_streams(
    logGroupName=TRAINING_LOG_GROUP,
    logStreamNamePrefix=training_job_name,
)['logStreams']

print(f'Log streams for {training_job_name}:')
for s in streams:
    print(f"  {s['logStreamName']}")

training_stream = streams[0]['logStreamName'] if streams else None

In [ ]:
# Read the beginning and end of the training log — startup and completion are the most informative parts
if training_stream:
    head = logs.get_log_events(
        logGroupName=TRAINING_LOG_GROUP, logStreamName=training_stream,
        startFromHead=True, limit=15,
    )['events']
    tail = logs.get_log_events(
        logGroupName=TRAINING_LOG_GROUP, logStreamName=training_stream,
        startFromHead=False, limit=15,
    )['events']

    def show(events, title):
        print(f'===== {title} =====')
        for e in events:
            ts = datetime.fromtimestamp(e['timestamp'] / 1000, tz=timezone.utc).strftime('%H:%M:%S')
            print(f"  {ts}  {e['message'][:140].rstrip()}")
        print()

    show(head, 'FIRST 15 LOG LINES (container + data setup)')
    show(tail, 'LAST 15 LOG LINES (completion / final metrics)')

In [ ]:
# 🔗 Open this training log in the console (second link has the ERROR search pre-filled)
if training_stream:
    show_links([
        ('Training log stream', url_log_group(TRAINING_LOG_GROUP, training_stream)),
        ('Training log stream — search "ERROR"', url_log_group(TRAINING_LOG_GROUP, training_stream, filter_pattern='ERROR')),
        ('Training log group (all jobs)', url_log_group(TRAINING_LOG_GROUP)),
    ])

In [ ]:
# Search the training logs for interesting patterns with FilterLogEvents
def search_logs(log_group, pattern, stream_prefix=None, hours_back=24*7, limit=25):
    """Search a log group for a filter pattern. Returns up to `limit` events.

    FilterLogEvents walks the time window in chunks and returns one page per
    chunk, so a page can come back EMPTY while still handing you a nextToken
    for a chunk that does contain matches. Reading only the first page is the
    classic way to "find" zero hits in a log that plainly contains them — the
    wider the window, the more likely it is. Hence the pagination loop.
    """
    kwargs = dict(
        logGroupName=log_group,
        filterPattern=pattern,
        startTime=int((datetime.now(timezone.utc) - timedelta(hours=hours_back)).timestamp() * 1000),
    )
    if stream_prefix:
        kwargs['logStreamNamePrefix'] = stream_prefix

    events, token = [], None
    while len(events) < limit:
        if token:
            kwargs['nextToken'] = token
        page = logs.filter_log_events(**kwargs)
        events.extend(page.get('events', []))
        token = page.get('nextToken')
        if not token:
            break
    return events[:limit]

# Patterns chosen to match what Lab 3A's train.py actually emits. A quoted
# pattern matches that exact substring; unquoted terms separated by spaces are
# ANDed, so 'MLflow run started' without quotes would mean something different.
for pattern in ['ERROR', '"Metrics:"', '"MLflow run started"']:
    events = search_logs(TRAINING_LOG_GROUP, pattern, stream_prefix=training_job_name)
    print(f'Pattern {pattern}: {len(events)} match(es)')
    for e in events[:5]:
        ts = datetime.fromtimestamp(e['timestamp'] / 1000, tz=timezone.utc).strftime('%m-%d %H:%M:%S')
        print(f'    {ts}  {e["message"][:120].rstrip()}')
    print()

print('Note the ERROR hit: it is pip complaining about dependency resolution during')
print('container setup, not a training failure. Grepping for a severity word finds')
print('every line that contains it — always read the match, never just the count.')

### Console alternative

You can browse the same logs interactively:

1. **SageMaker Console → Training → Training jobs → (your job) → Monitor → View logs**, or
2. **CloudWatch Console → Logs → Log groups → `/aws/sagemaker/TrainingJobs`** → click your job's stream → use the 🔍 search box (`ERROR`, `accuracy`, ...). Use **Actions → Download search results** to export for offline analysis or sharing.


## Section 3: Monitor Endpoint Logs in (Near) Real-Time

Endpoint logs live in `/aws/sagemaker/Endpoints/<EndpointName>` — one stream per instance (`AllTraffic/i-...`). They record model-server startup, health pings, each scoring request, and errors.

We'll send traffic and then read what the container logged.


In [ ]:
# Send a burst of valid requests. 20 label-encoded features in schema order,
# no header and no target; pdays=999 means "never previously contacted".
test_csv = '56,1,1,1,0,0,0,1,4,2,180,2,999,0,1,-1.8,92.893,-46.2,1.299,5099.1'

if endpoint_name:
    traffic_start = datetime.now(timezone.utc)
    for i in range(5):
        resp = smr.invoke_endpoint(EndpointName=endpoint_name, ContentType='text/csv', Body=test_csv)
        body = json.loads(resp['Body'].read().decode())
        print(f'request {i+1}/5 → class={body["predictions"][0]} '
              f'P(subscribe)={body["probabilities"]["yes"][0]:.4f}')
        time.sleep(1)
    print('\n✅ Traffic sent — logs propagate within a few seconds to a minute.')

In [ ]:
# Read the most recent endpoint log events
if endpoint_name:
    time.sleep(20)  # give logs a moment to propagate
    streams = logs.describe_log_streams(
        logGroupName=ENDPOINT_LOG_GROUP, orderBy='LastEventTime', descending=True,
    )['logStreams']
    print(f'Endpoint log streams: {[s["logStreamName"] for s in streams]}\n')

    events = logs.get_log_events(
        logGroupName=ENDPOINT_LOG_GROUP,
        logStreamName=streams[0]['logStreamName'],
        startFromHead=False, limit=20,
    )['events']

    print('Last 20 endpoint log lines:')
    for e in events:
        ts = datetime.fromtimestamp(e['timestamp'] / 1000, tz=timezone.utc).strftime('%H:%M:%S')
        print(f'  {ts}  {e["message"][:130].rstrip()}')

## Section 4: Trigger and Diagnose a Real Inference Error

The fastest way to learn log-based debugging is to break something on purpose — and the first attempt here teaches something the lab did not plan for.

The Lab 3A handler is deliberately forgiving: it coerces unparseable values to `NaN` and fills any feature the request left out with `0.0`. A payload with the **wrong number of features** therefore comes back `200 OK`, carrying a prediction computed from mostly-default inputs. Nothing lands in the endpoint's error metrics. That is the failure mode worth knowing about — bad input that looks perfectly healthy.

To get a genuine server-side exception you have to break the request *before* the model is reached. An **unparseable body** does it: `input_fn` calls `json.loads` first and rejects it outright. That one gives you the client-side error and the stack trace to chase through the logs.


In [ ]:
# Two kinds of bad request, two very different outcomes.
if endpoint_name:
    error_time = datetime.now(timezone.utc)

    # 1. Wrong shape: 5 values where the model expects 20. The handler coerces
    #    the non-numeric strings to NaN and fills the 15 absent features with
    #    0.0, so this returns 200 with a prediction built from mostly defaults.
    try:
        resp = smr.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='text/csv',
            Body='invalid,data,wrong,feature,count',
        )
        print('Wrong feature count -> 200 OK, silently absorbed:')
        print(f'  {resp["Body"].read().decode()[:200]}')
    except Exception as e:
        print(f'Wrong feature count -> {type(e).__name__}: {str(e)[:200]}')

    # 2. Unparseable body: input_fn calls json.loads before the model is ever
    #    reached, so this raises server-side and surfaces as a real 4XX/5XX.
    try:
        smr.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body='{this is not json',
        )
        print('\nUnparseable JSON -> 200 OK (unexpected — check the handler)')
    except Exception as e:
        print('\nUnparseable JSON -> client-side error (as expected):')
        print(f'  {type(e).__name__}: {str(e)[:300]}')

In [ ]:
# Find the corresponding server-side error in the endpoint logs
if endpoint_name:
    time.sleep(20)  # log propagation
    events = search_logs(ENDPOINT_LOG_GROUP, '?ERROR ?Error ?error ?Exception', hours_back=1)
    if events:
        print(f'Found {len(events)} error-related log line(s):\n')
        for e in events[-10:]:
            ts = datetime.fromtimestamp(e['timestamp'] / 1000, tz=timezone.utc).strftime('%H:%M:%S')
            print(f'  {ts}  {e["message"][:160].rstrip()}')
    else:
        print('No error lines found yet — logs can lag; re-run this cell in ~30 seconds.')

    print()
    print('💡 This is the debugging loop: client 4XX/5XX → endpoint log group → stack trace →')
    print('   root cause (here: an unparseable JSON body rejected by input_fn).')

In [ ]:
# 🔗 Inspect the error in the console — search filter pre-filled
if endpoint_name:
    show_links([
        ('Endpoint log group', url_log_group(ENDPOINT_LOG_GROUP)),
        ('Latest endpoint stream — search "Error"',
         url_log_group(ENDPOINT_LOG_GROUP, streams[0]['logStreamName'], filter_pattern='Error')),
    ])

## Section 5: Programmatic Log Analysis

`filter_log_events` lets you build lightweight log analytics without any extra infrastructure — error frequency, timelines, and pattern extraction.


In [ ]:
# Error frequency analysis over the last hour, bucketed per 5 minutes
if endpoint_name:
    all_events = search_logs(ENDPOINT_LOG_GROUP, '', hours_back=1, limit=500)   # empty pattern = all events
    err_events = [e for e in all_events if any(k in e['message'] for k in ('ERROR', 'Error', 'Exception'))]

    print(f'Total log events (last hour): {len(all_events)}')
    print(f'Error events                : {len(err_events)}')

    if all_events:
        df = pd.DataFrame([{
            'ts': datetime.fromtimestamp(e['timestamp'] / 1000, tz=timezone.utc),
            'is_error': any(k in e['message'] for k in ('ERROR', 'Error', 'Exception')),
        } for e in all_events])
        df['bucket'] = df['ts'].dt.floor('5min')
        summary = df.groupby('bucket').agg(events=('is_error', 'size'), errors=('is_error', 'sum'))
        print('\nEvents per 5-minute bucket:')
        print(summary.to_string())

In [ ]:
# Extract and count distinct error types (poor-man's error classification)
if endpoint_name and err_events:
    import re
    def classify(msg):
        m = re.search(r'([A-Za-z]+(?:Error|Exception))', msg)
        return m.group(1) if m else 'UnclassifiedError'
    counts = Counter(classify(e['message']) for e in err_events)
    print('Error types (last hour):')
    for etype, n in counts.most_common():
        print(f'  {etype:30s} {n}')

## Section 6: Logs Insights, Metric Filters, and Log-Based Alarms (Console)

These three features turn logs into actionable monitoring. They require write permissions your notebook role doesn't have, so do them in the **console** — the exact steps and copy-paste snippets are below.

### 6.1 CloudWatch Logs Insights

**Console → CloudWatch → Logs → Logs Insights** → select log group `/aws/sagemaker/Endpoints/<your-endpoint>` → time range *Last 1 hour*. Try these queries:

```sql
-- 100 most recent events
fields @timestamp, @message
| sort @timestamp desc
| limit 100
```

```sql
-- errors only
fields @timestamp, @message
| filter @message like /(?i)(error|exception)/
| sort @timestamp desc
| limit 50
```

```sql
-- request volume per 5 minutes (capacity planning)
fields @timestamp
| stats count() as request_count by bin(5m)
| sort @timestamp desc
```

```sql
-- group errors by type
fields @message
| filter @message like /ERROR/
| parse @message /(?<error_type>[A-Za-z]+(Error|Exception))/
| stats count() by error_type
| sort count desc
```

💾 After running a useful query: **Actions → Save** (e.g. `Endpoint-Error-Analysis`) so your team can reuse it.

### 6.2 Metric filter: count errors as a metric

**Console → CloudWatch → Logs → Log groups → your endpoint log group → Actions → Create metric filter**:

| Setting | Value |
|---|---|
| Filter pattern | `?ERROR ?Exception` |
| Filter name | `EndpointErrors` |
| Metric namespace | `SageMaker/CustomMetrics` |
| Metric name | `ErrorCount` |
| Metric value / default | `1` / `0` |

### 6.3 Alarm on the log-based metric

**Console → CloudWatch → Alarms → Create alarm** → select `SageMaker/CustomMetrics → ErrorCount`:

- Statistic **Sum**, period **5 minutes**, threshold **> 10**
- *Treat missing data as* **notBreaching** (idle endpoints emit no datapoints)
- Notification → existing SNS topic **`SageMaker-Alerts`** (from Lab 5A)
- Name: `High-Endpoint-Error-Count`

**Test it**: run the cell below to fire >10 malformed requests, then watch the alarm transition to `ALARM` within ~5-10 minutes.


In [ ]:
# 🔗 Prefilled Logs Insights links — each opens the console with query AND log group already set
if endpoint_name:
    q_recent = 'fields @timestamp, @message\n| sort @timestamp desc\n| limit 100'
    q_errors = 'fields @timestamp, @message\n| filter @message like /(?i)(error|exception)/\n| sort @timestamp desc\n| limit 50'
    q_volume = 'fields @timestamp\n| stats count() as request_count by bin(5m)\n| sort @timestamp desc'
    show_links([
        ('Logs Insights — 100 most recent events', url_logs_insights(q_recent, ENDPOINT_LOG_GROUP)),
        ('Logs Insights — errors only', url_logs_insights(q_errors, ENDPOINT_LOG_GROUP)),
        ('Logs Insights — request volume per 5 min', url_logs_insights(q_volume, ENDPOINT_LOG_GROUP)),
        ('Metric filter setup — endpoint log group page (→ Actions → Create metric filter)',
         url_log_group(ENDPOINT_LOG_GROUP)),
    ])
    print('Tip: just hit "Run query" once the page opens — everything is pre-filled.')

In [ ]:
# Generate 15 malformed requests to exercise the log-based alarm created in the console.
# Unparseable JSON, not a short CSV row: only the former actually errors
# server-side, and an alarm on absorbed input would never fire.
if endpoint_name:
    for i in range(15):
        try:
            smr.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType='application/json',
                Body='{this is not json',
            )
        except Exception:
            pass
        print(f'  malformed request {i+1}/15 sent')
        time.sleep(1)
    print('\n✅ Done. If you created the EndpointErrors metric filter + alarm, expect')
    print('   ALARM state (and an SNS email, if confirmed) within ~5-10 minutes.')

## Section 7: Structured Logging Best Practices

The lab endpoint logs whatever the XGBoost container prints. In **your own inference code** (custom containers / `inference.py`), structured logging makes every technique in this lab dramatically more effective:

**1. Log JSON, not prose** — trivially parseable by Logs Insights:

```python
import json, logging
logger = logging.getLogger(__name__)

logger.info(json.dumps({
    "event": "prediction_completed",
    "request_id": request_id,          # 2. always include correlation IDs
    "latency_ms": latency,             # 3. log performance numbers
    "model_version": "v1.2.3",         # 4. log context (versions, shapes)
    "input_rows": len(input_data),
}))
```

**5. Use log levels properly** — `DEBUG` (verbose diagnostics) / `INFO` (normal ops) / `WARNING` (potential issues) / `ERROR` (failures) / `CRITICAL` (page someone).

**6. Sample high-volume logs** — on busy endpoints, log full payload details for only ~10% of requests (`if random.random() < 0.1`), errors always.

With JSON logs, Logs Insights queries become surgical:

```sql
fields @timestamp, latency_ms, model_version
| filter event = "prediction_completed" and latency_ms > 100
| stats avg(latency_ms), max(latency_ms) by model_version
```


## Section 8: Log Retention and Archival

By default SageMaker log groups **never expire** — fine for a workshop, expensive at scale.

**Set retention (console)**: CloudWatch → Logs → Log groups → select group → **Actions → Edit retention setting**:

| Environment | Suggested retention |
|---|---|
| Development | 7–14 days |
| Production | 30–90 days |
| Regulated / compliance | 1–7 years (or export to S3) |

**Archive to S3 for long-term/compliance**: log group → **Actions → Export data to Amazon S3** (one-off), or automate with a scheduled Lambda calling `logs.create_export_task(...)`. Exported objects can then ride S3 lifecycle policies down to Glacier ($0.004/GB-month vs $0.03/GB-month in CloudWatch Logs).


## Troubleshooting Cheat Sheet

| Symptom | Where to look | Common root causes |
|---|---|---|
| Training job fails in <2 min | First lines of training log stream | S3 permissions, bad image/entry point, script syntax error |
| Endpoint 5XX errors | Endpoint log group, search `ERROR` | Model load failure, OOM, timeout |
| Endpoint 4XX errors | Client exception + endpoint logs | Malformed payload, wrong `ContentType`, feature-count mismatch (you saw this in Section 4!) |
| Latency spikes | Logs around the spike timestamp + Lab 5A metrics | Cold start (model reload), GC pauses, oversized inputs |
| No logs at all | `describe_log_streams` empty | Wrong region, endpoint never invoked, container crashed pre-logging |

**Method**: metric anomaly (Lab 5A) → narrow the time window → read logs in that window (this lab) → root cause.

## Key Takeaways

✅ Training logs: `/aws/sagemaker/TrainingJobs`, one stream per instance — first/last lines tell most of the story.
✅ Endpoint logs: `/aws/sagemaker/Endpoints/<name>` — the place to find server-side stack traces behind 4XX/5XX.
✅ `filter_log_events` enables programmatic pattern search and lightweight analytics from a notebook.
✅ **Logs Insights** (console) scales the same analysis with a SQL-like language; save your queries.
✅ **Metric filters + alarms** convert log patterns into proactive notifications.
✅ Structured (JSON) logging with request IDs makes production debugging tractable.
✅ Retention policies and S3 archival balance compliance with cost.

## Next Steps

Continue to **Lab 5C: CloudTrail Logs Monitoring** (`lab5c-cloudtrail-monitoring.ipynb`) to audit *who did what* to your ML resources — API-level governance and compliance tracking.
